# 17 -- Checkpoint/resume validation: extend a real scan, compare against a fresh one

Validates the checkpointing scheme the way it's actually used in production.
Self-contained -- builds everything itself via `ptbuilder.ic.generate_surrogate`,
no round-tripping through other notebooks (notebook 15 has its own,
not-yet-migrated copy of this scan-building logic; this notebook doesn't
touch or depend on it).

**Workflow:**
1. Run **Section 1** to build two independent scans -- `gs1-4` and `gs1-5`
   (the ground truth) -- and write both their SLURM scripts. Submit both
   (they can run concurrently), wait for both to finish.
2. Run **Section 2** to extend the `gs1-4` scan to `gs1-5` in place (Format
   A -- each row's own embedded plateau checkpoint seeds it, no
   re-integration from `s=0`) and write its SLURM script. Submit, wait.
3. Run **Section 3** to compare a few representative rows between the
   extended scan and the independently-built ground truth at their shared
   `T_MAX`.

The ground-truth scan uses a distinct tag (`lb0.84_direct` vs `lb0.84`), so
it never collides with the extended scan's directory -- no rename-out-of-
the-way step needed.

The two scans' `wlist` grids differ (the extended scan keeps `gs1-4`'s
original frequency grid; the independently-built `gs1-5` scan has its own,
anchored to a different `GAMMA_STAR_MAX`) -- that's expected, not a bug.
Section 3 plots both curves together on the same axes rather than requiring
matching grids: if checkpointing is correct, they should trace the same
spectrum regardless of where each one happens to sample it.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

import ptbuilder as pt
from ptbuilder.analysis import load_scan, query_spectrum
from ptbuilder.ic import generate_surrogate
from ptbuilder.physics import Phi4Potential, PhysicsModel

ROOT = Path(pt.__file__).parent.parent
LAMBDA_BAR = 0.84

model = PhysicsModel(Phi4Potential(LAMBDA_BAR))

GAMMA_STAR_MIN = 1.0
KR_MIN, KR_MAX, N_OMEGA = 1.0, 100.0, 32   # production values

GAMMA_IJ_TARGETS = [1.0, 2.5, 5.0]

def scan_path(tag_label, gamma_star_max):
    tag = f'{tag_label}_gs{GAMMA_STAR_MIN:g}-{gamma_star_max:g}_kR{KR_MIN:g}-{KR_MAX:g}_nw{N_OMEGA}'
    return ROOT / 'data' / f'runtime_scan_{tag}'

## 1. Build gs1-4 and gs1-5 (independent, can run concurrently)

`gs1-4` uses `--overwrite` since whatever is on disk under that name
predates this session's checkpoint feature and needs regenerating with the
current binary (no `plateau_re` embedded otherwise). `gs1-5` is a normal
fresh build -- the ground truth Section 3 validates against.

In [ ]:
def write_slurm_script(scan, gpu_walltime='04:00:00', gpu_nodes=2, force_overwrite=False,
                       gpu_account='m3166_g'):
    """Writes scripts/runtime_scan_{tag}_gpu.sh for `scan` (from
    generate_surrogate) -- same xargs -P worker-pool pattern as notebook 15,
    duplicated here rather than shared since this is cluster-specific.

    Dispatches rows LARGEST-gamma_ij-first (descending), not ascending:
    runtimes vary by orders of magnitude across the grid, so starting with
    the few slow, high-gamma_ij rows and backfilling the many fast, cheap
    ones behind them keeps the worker pool busy throughout -- ascending
    order does the opposite (front-loads cheap rows, then leaves most of the
    pool idle at the end waiting on the last few slow ones). Row numbering/
    naming is untouched -- only the ORDER xargs -P pulls them in changes."""
    (ROOT / 'logs').mkdir(exist_ok=True)
    gpus_per_node = 4
    n_concurrent  = gpu_nodes * gpus_per_node
    n_tasks       = len(scan.manifest_rows)
    relative_manifest = scan.manifest.relative_to(ROOT)
    relative_output   = scan.output_root.relative_to(ROOT)
    overwrite_flag = ' --overwrite' if force_overwrite else ''

    script_path = ROOT / 'scripts' / f'runtime_scan_{scan.tag}_gpu.sh'
    script_path.write_text(f'''#!/bin/bash
#SBATCH --job-name=bm_{scan.tag}_gpu
#SBATCH --output=logs/bm_{scan.tag}_gpu_%j.out
#SBATCH --constraint=gpu
#SBATCH --qos=regular
#SBATCH -A {gpu_account}
#SBATCH --nodes={gpu_nodes}
#SBATCH --ntasks={n_concurrent}
#SBATCH --ntasks-per-node={gpus_per_node}
#SBATCH --cpus-per-task=32
#SBATCH --gpus-per-task=1
#SBATCH --time={gpu_walltime}
set -euo pipefail
REPO_ROOT="${{SLURM_SUBMIT_DIR:-$PWD}}"
cd "$REPO_ROOT"

run_one() {{
    local row_num=$1
    local line
    line=$(sed -n "$((row_num + 1))p" {relative_manifest})
    IFS=$'\t' read -r index gamma n_t setup name <<< "$line"
    output="{relative_output}/${{name}}"
    mkdir -p "$output"
    /usr/bin/time --verbose srun --exclusive --exact -N 1 -n 1 -c 32 --gpus-per-task=1 --cpu-bind=cores bin/bubblemaster_gpu "$setup" "$output/" --param {scan.n_min} --filon-panels-per-osc {scan.panels_per_oscillation} --s-batch-size {scan.s_batch_size}{overwrite_flag}
}}
export -f run_one

seq {n_tasks} -1 1 | xargs -P {n_concurrent} -I ROW bash -c 'run_one "$@"' _ ROW
''')
    script_path.chmod(0o755)
    print(f'Written: {script_path.relative_to(ROOT)}  '
          f'({n_tasks} tasks, {n_concurrent} concurrent slots on {gpu_nodes} nodes)'
          + (', --overwrite' if force_overwrite else ''))
    return script_path


scan_4        = generate_surrogate(model, ROOT, 'lb0.84', GAMMA_STAR_MIN, 4.0,
                                   KR_MIN, KR_MAX, N_OMEGA)
scan_5_direct = generate_surrogate(model, ROOT, 'lb0.84_direct', GAMMA_STAR_MIN, 5.0,
                                   KR_MIN, KR_MAX, N_OMEGA)

script_4        = write_slurm_script(scan_4, force_overwrite=True)
script_5_direct = write_slurm_script(scan_5_direct, force_overwrite=False)

print()
print('Submit both (can run concurrently):')
print(f'  sbatch {script_4.relative_to(ROOT)}')
print(f'  sbatch {script_5_direct.relative_to(ROOT)}')

## 2. Extend gs1-4 -> gs1-5

Run after both of Section 1's SLURM jobs finish. Renames the `gs1-4` scan
directory to `gs1-5`, extends its existing rows' `times[]` in place using
each row's own embedded plateau checkpoint, and computes any newly-in-range
rows fresh -- no `--overwrite` needed here, the GPU binary auto-detects
what's already done from `output_dir/` contents.

In [ ]:
scan_5_extended = generate_surrogate(model, ROOT, 'lb0.84', GAMMA_STAR_MIN, 5.0,
                                     KR_MIN, KR_MAX, N_OMEGA, extend_from=scan_4.scan_root)
script_5_extended = write_slurm_script(scan_5_extended, force_overwrite=False)
print(f'Submit: sbatch {script_5_extended.relative_to(ROOT)}')

## 3. Compare both gs1-5 scans at exactly gamma_ij = 1, 2.5, 5

Uses `ptbuilder.analysis.query_spectrum` to evaluate each scan's surrogate
at the exact requested `gamma_ij` (log-linear interpolation between the two
bracketing rows -- the same domain-aware method already validated for the
N=64 reconstruction comparison in `16_n64_reconstruction_scan_comparison.ipynb`),
rather than comparing whichever rows happen to be nearest. Both scans are
queried at their shared `T_MAX`.

Loads scan paths from disk (not the in-memory `scan_*` objects from
Sections 1-2) -- this section is meant to be runnable on its own, e.g.
after a kernel restart while waiting on the cluster.

In [ ]:
DIRECT_SCAN   = scan_path('lb0.84_direct', 5.0)
EXTENDED_SCAN = scan_path('lb0.84', 5.0)

for path, label in [(DIRECT_SCAN, 'direct'), (EXTENDED_SCAN, 'extended')]:
    if not path.exists():
        raise FileNotFoundError(f'{path} not found -- see the workflow steps in the intro '
                                f'markdown cell ({label} scan missing)')

scan_direct   = load_scan(DIRECT_SCAN)      # reads only whatever result_*.h5 files exist
scan_extended = load_scan(EXTENDED_SCAN)

t_max_direct   = scan_direct.rows[0].times[-1]
t_max_extended = scan_extended.rows[0].times[-1]
assert abs(t_max_direct - t_max_extended) < 1e-6 * max(1.0, abs(t_max_direct)), (
    f'direct T_MAX={t_max_direct:.6f} != extended T_MAX={t_max_extended:.6f} -- '
    f'both scans should share the same global T_MAX')
T_MAX = t_max_direct

fig, axes = plt.subplots(1, len(GAMMA_IJ_TARGETS), figsize=(5*len(GAMMA_IJ_TARGETS), 4))
if len(GAMMA_IJ_TARGETS) == 1:
    axes = [axes]

for ax, gamma_target in zip(axes, GAMMA_IJ_TARGETS):
    k_d, spec_d, _ = query_spectrum(scan_direct, gamma_target, T_MAX)
    k_e, spec_e, _ = query_spectrum(scan_extended, gamma_target, T_MAX)

    ax.plot(k_d, spec_d, 'o-', color='k', lw=1.5, ms=3, label='direct')
    ax.plot(k_e, spec_e, 's--', color='tab:red', lw=1.5, ms=3, label='extended')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('w')
    ax.set_title(f'gamma_ij={gamma_target:g}  (t={T_MAX:.2f})')
    ax.legend(frameon=False, fontsize=8)

axes[0].set_ylabel('spectrum')
plt.tight_layout()
plt.savefig(ROOT / 'checkpoint_validation_scan_extend.pdf', bbox_inches='tight')
plt.show()